# 14 - EF Regression Grad-CAM

Initial EF Grad-CAM extraction for the notebook-12 EF-primary ConvLSTM models. This notebook only performs standard Grad-CAM extraction and visualization. It does not run temporal evaluation, optical-flow comparison, perturbation testing, or quantitative XAI metrics.

## Setup

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import os
import sys
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm


def first_existing_path(candidates):
    cleaned = [candidate for candidate in candidates if candidate]
    for candidate in cleaned:
        path = Path(candidate)
        if path.exists():
            return path
    return Path(cleaned[-1])


PROJECT_ROOT = first_existing_path([
    os.environ.get("PROJECT_ROOT"),
    "/kaggle/input/echonet-temporal-xai",
    "/kaggle/input/src-updated",
    "/kaggle/input/datasets/jiyoonoh24/echonet-src-code",
    "/kaggle/working/Echonet_temporal_XAI",
    Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd(),
])
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import EchoNetTemporalDataset, load_temporal_metadata, split_by_echonet_filelist
from src.gradcam_ef_regression import (
    EchoNetTemporalEFDataset,
    build_ef_regression_convlstm,
    encoder_bottleneck_ef_gradcam,
    load_exact_checkpoint,
    make_gradcam_overlay_figure,
    make_motion_trace_overlay_figure,
    make_centroid_trajectory_plot,
    make_temporal_diagnostic_plot,
    run_normal_inference,
    save_gradcam_npz,
    select_representative_samples,
    temporal_representation_ef_probe_gradcam,
)
from src.utils import load_echonet_tables, set_seed

RAW_DIR = Path(os.environ.get("ECHONET_RAW_DIR", PROJECT_ROOT / "data" / "raw" / "EchoNet-Dynamic"))
PROCESSED_DIR = Path(os.environ.get("ECHONET_PROCESSED_DIR", PROJECT_ROOT / "data" / "processed"))
VIDEOS_DIR = RAW_DIR / "Videos"

TRAINED_RUN_DIR = Path(os.environ.get(
    "EF_MOTION_CONVLSTM_RUN_DIR",
    "/kaggle/input/ef-primary-motion-head-conv-lstm-07-22" if Path("/kaggle/input").exists() else PROJECT_ROOT / "outputs" / "runs" / "ef_primary_motion_head_conv_lstm_07_22",
))

RUN_DIR = Path(os.environ.get(
    "EF_GRADCAM_RUN_DIR",
    "/kaggle/working/outputs/runs/ef_gradcam" if Path("/kaggle/working").exists() else PROJECT_ROOT / "outputs" / "runs" / "ef_gradcam",
))
MANIFEST_DIR = RUN_DIR / "manifests"
for directory in [RUN_DIR, MANIFEST_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw EchoNet directory: {RAW_DIR}")
print(f"Processed directory: {PROCESSED_DIR}")
print(f"Trained ConvLSTM EF run: {TRAINED_RUN_DIR}")
print(f"Grad-CAM output directory: {RUN_DIR}")


## Configuration

In [ ]:
RUN_MODE = "smoke"  # change to "full" for 10 samples per model

BASE_CONFIG = {
    "seed": 42,
    "num_frames_before": 11,
    "num_frames_after": 11,
    "temporal_stride": 2,
    "target_idx": 11,
    "sequence_length": 23,
    "image_size": [112, 112],
    "channels": [16, 32, 64, 128],
    "ef_hidden_dim": 128,
    "dropout": 0.1,
    "motion_hidden_channels": 64,
    "batch_size_for_prediction": 4,
    "num_workers": 2,
    "smoke_gradcam_samples_per_model": 2,
    "full_gradcam_samples_per_model": 10,
    "overlay_alpha": 0.45,
}

run_config_path = TRAINED_RUN_DIR / "config.json"
if run_config_path.exists():
    trained_config = json.loads(run_config_path.read_text())
    for key in [
        "num_frames_before", "num_frames_after", "temporal_stride", "target_idx", "sequence_length",
        "image_size", "channels", "ef_hidden_dim", "dropout", "motion_hidden_channels",
    ]:
        if key in trained_config:
            BASE_CONFIG[key] = trained_config[key]

GRADCAM_SAMPLE_COUNT = BASE_CONFIG["smoke_gradcam_samples_per_model"] if RUN_MODE == "smoke" else BASE_CONFIG["full_gradcam_samples_per_model"]
set_seed(int(BASE_CONFIG["seed"]))
with (RUN_DIR / "config.json").open("w", encoding="utf-8") as f:
    json.dump({**BASE_CONFIG, "run_mode": RUN_MODE, "gradcam_sample_count": GRADCAM_SAMPLE_COUNT}, f, indent=2)
BASE_CONFIG


## Load Official Test Dataset

In [ ]:
metadata_path = PROCESSED_DIR / "metadata.csv"
assert metadata_path.exists(), f"Missing processed metadata: {metadata_path}"
assert (RAW_DIR / "FileList.csv").exists(), f"Missing FileList.csv under {RAW_DIR}"
assert VIDEOS_DIR.exists(), f"Missing Videos directory: {VIDEOS_DIR}"

samples = load_temporal_metadata(metadata_path)
file_list, _volume_tracings = load_echonet_tables(RAW_DIR)
assert "EF" in file_list.columns, "FileList.csv must contain EF labels."

echo_table = file_list.copy()
echo_table["video_stem"] = echo_table["FileName"].astype(str).map(lambda x: Path(x).stem)
ef_lookup = dict(zip(echo_table["video_stem"], echo_table["EF"].astype(float)))

samples_with_ef = []
for sample in samples:
    item = dict(sample)
    video_stem = Path(str(item["video_id"])).stem
    if video_stem in ef_lookup and pd.notna(ef_lookup[video_stem]):
        item["ef"] = float(ef_lookup[video_stem])
        samples_with_ef.append(item)

train_samples, val_samples, test_samples = split_by_echonet_filelist(samples_with_ef, file_list)
ef_values_train = np.array([float(sample["ef"]) for sample in train_samples], dtype=np.float32)
ef_mean = float(ef_values_train.mean())
ef_std = float(ef_values_train.std(ddof=0))
assert ef_std > 0, "Training EF standard deviation is zero; cannot normalize EF."

base_test_dataset = EchoNetTemporalDataset(
    test_samples,
    videos_dir=VIDEOS_DIR,
    num_frames_before=int(BASE_CONFIG["num_frames_before"]),
    num_frames_after=int(BASE_CONFIG["num_frames_after"]),
    temporal_stride=int(BASE_CONFIG["temporal_stride"]),
    image_size=tuple(BASE_CONFIG["image_size"]),
    augment=False,
)
test_dataset = EchoNetTemporalEFDataset(base_test_dataset, ef_mean=ef_mean, ef_std=ef_std)
prediction_loader = DataLoader(
    test_dataset,
    batch_size=int(BASE_CONFIG["batch_size_for_prediction"]),
    shuffle=False,
    num_workers=int(BASE_CONFIG["num_workers"]),
    pin_memory=torch.cuda.is_available(),
)

gradcam_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)
sample_batch = next(iter(gradcam_loader))
assert tuple(sample_batch["sequence"].shape[1:]) == (int(BASE_CONFIG["sequence_length"]), 1, *tuple(BASE_CONFIG["image_size"])), tuple(sample_batch["sequence"].shape)
print(f"Train/test samples with EF: {len(train_samples):,} / {len(test_samples):,}")
print(f"EF normalization: mean={ef_mean:.3f}, std={ef_std:.3f}")
print(f"One sequence: {tuple(sample_batch['sequence'].shape)}")


## Recreate Models and Load Best Checkpoints

In [ ]:
checkpoint_paths = {
    "ef_primary": Path(os.environ.get(
        "EF_PRIMARY_CHECKPOINT_PATH",
        TRAINED_RUN_DIR / "checkpoints" / "ef_primary" / "best_ef_mae.pt",
    )),
    "ef_primary_motion": Path(os.environ.get(
        "EF_PRIMARY_MOTION_CHECKPOINT_PATH",
        TRAINED_RUN_DIR / "checkpoints" / "ef_primary_motion" / "best_ef_mae.pt",
    )),
}
for name, path in checkpoint_paths.items():
    assert path.exists(), f"Missing {name} checkpoint: {path}"

model_specs = {
    "ef_primary": {"with_motion": False},
    "ef_primary_motion": {"with_motion": True},
}
models = {}
checkpoint_metadata = {}
for model_name, spec in model_specs.items():
    model = build_ef_regression_convlstm(BASE_CONFIG, with_motion=bool(spec["with_motion"])).to(device)
    metadata = load_exact_checkpoint(model, checkpoint_paths[model_name], device="cpu")
    model.to(device)
    model.eval()
    models[model_name] = model
    checkpoint_metadata[model_name] = metadata
    print(model_name, json.dumps({k: v for k, v in metadata.items() if k not in {"config", "metrics"}}, indent=2))

with (MANIFEST_DIR / "checkpoint_metadata.json").open("w", encoding="utf-8") as f:
    json.dump(checkpoint_metadata, f, indent=2, default=str)


## Normal Checkpoint Inference and Sample Selection

In [ ]:
prediction_tables = {}
dataset_metrics = {}
for model_name, model in models.items():
    predictions, metrics = run_normal_inference(model, prediction_loader, device, ef_mean, ef_std, model_name=model_name)
    prediction_tables[model_name] = predictions
    dataset_metrics[model_name] = metrics
    predictions.to_csv(MANIFEST_DIR / f"{model_name}_normal_test_predictions.csv", index=False)
    print(model_name, json.dumps(metrics, indent=2))

with (MANIFEST_DIR / "normal_test_metrics.json").open("w", encoding="utf-8") as f:
    json.dump(dataset_metrics, f, indent=2)

selected_rows = []
selected_by_model = {}
for model_name, predictions in prediction_tables.items():
    selected = select_representative_samples(predictions, count=GRADCAM_SAMPLE_COUNT)
    selected_by_model[model_name] = selected
    for rank, sample_id in enumerate(selected):
        row = predictions[predictions["sample_id"] == sample_id].iloc[0].to_dict()
        row["selection_rank"] = rank
        selected_rows.append(row)
selected_df = pd.DataFrame(selected_rows)
selected_df.to_csv(MANIFEST_DIR / "selected_gradcam_samples.csv", index=False)
display(selected_df)


## Smoke-Test Grad-CAM Diagnostics


In [ ]:
def print_gradcam_smoke_diagnostics(model_name: str, cam_type: str, result, normal_prediction: float, diagnostics_csv_path: Path) -> None:
    diagnostics = result.temporal_diagnostics
    gradient_abs_mean = diagnostics["gradient_abs_mean"].to_numpy()
    positive_cam_max = diagnostics["positive_cam_max"].to_numpy()
    abs_signed = np.abs(result.signed_raw_cams).astype(np.float64)
    assert np.isfinite(result.signed_raw_cams).all(), f"{model_name} {cam_type}: non-finite signed_raw_cams"
    assert np.isfinite(result.frame_normalized_positive_cams).all(), f"{model_name} {cam_type}: non-finite frame_normalized_positive_cams"
    assert float(abs_signed.max()) > 1e-12, f"{model_name} {cam_type}: signed_raw_cams are all zero"
    assert float(result.frame_normalized_positive_cams.max()) > 1e-12, f"{model_name} {cam_type}: frame_normalized_positive_cams are all zero"
    zero_positive_frames = int((positive_cam_max <= 1e-12).sum())
    near_zero_gradient_frames = int((gradient_abs_mean <= 1e-12).sum())
    diagnostics_csv_path.parent.mkdir(parents=True, exist_ok=True)
    diagnostics.to_csv(diagnostics_csv_path, index=False)
    print(f"\n[{model_name} | {cam_type}]")
    print(f"normal inference EF:       {normal_prediction:.6f}")
    print(f"Grad-CAM forward EF:       {result.pred_ef:.6f}")
    print(f"prediction difference:     {abs(result.pred_ef - normal_prediction):.8f}")
    print(f"max absolute signed CAM:   {float(abs_signed.max()):.8e}")
    print(f"mean absolute signed CAM:  {float(abs_signed.mean()):.8e}")
    print(f"median abs signed CAM:     {float(np.median(abs_signed)):.8e}")
    print(f"95th pct abs signed CAM:   {float(np.percentile(abs_signed, 95)):.8e}")
    print(f"99th pct abs signed CAM:   {float(np.percentile(abs_signed, 99)):.8e}")
    print("number of zero-positive frames:", zero_positive_frames)
    print("number of near-zero-gradient frames:", near_zero_gradient_frames)
    print("per-timestep diagnostics CSV:", diagnostics_csv_path)


sample_id_to_index = {str(sample["id"]): idx for idx, sample in enumerate(test_dataset.base_dataset.samples)}

for model_name, model in models.items():
    smoke_sample_id = selected_by_model[model_name][0]
    smoke_idx = sample_id_to_index[smoke_sample_id]
    smoke_batch = next(iter(DataLoader(Subset(test_dataset, [smoke_idx]), batch_size=1, shuffle=False, num_workers=0)))
    smoke_sequence = smoke_batch["sequence"].to(device)
    normal_prediction = float(prediction_tables[model_name][prediction_tables[model_name]["sample_id"] == smoke_sample_id].iloc[0]["ef_pred"])

    smoke_encoder = encoder_bottleneck_ef_gradcam(model, smoke_sequence, ef_mean=ef_mean, ef_std=ef_std)
    print_gradcam_smoke_diagnostics(
        model_name,
        "encoder_bottleneck",
        smoke_encoder,
        normal_prediction,
        MANIFEST_DIR / "smoke_diagnostics" / f"{model_name}_{smoke_sample_id}_encoder_bottleneck_diagnostics.csv",
    )

    smoke_temporal = temporal_representation_ef_probe_gradcam(model, smoke_sequence, ef_mean=ef_mean, ef_std=ef_std)
    print_gradcam_smoke_diagnostics(
        model_name,
        "temporal_representation",
        smoke_temporal,
        normal_prediction,
        MANIFEST_DIR / "smoke_diagnostics" / f"{model_name}_{smoke_sample_id}_temporal_representation_diagnostics.csv",
    )


## Generate EF Grad-CAM NPZ Files and Overlays

In [ ]:
sample_id_to_index = {str(sample["id"]): idx for idx, sample in enumerate(test_dataset.base_dataset.samples)}
manifest_rows = []
all_diagnostic_rows = []
all_centroid_rows = []

overlay_specs = [
    {
        "suffix": "positive_frame_normalized_primary",
        "cam_key": "frame_normalized_positive_cams",
        "overlay_name": "positive frame-normalized (primary)",
        "signed": False,
        "positive_display_mode": "enhanced",
        "cmap_name": "turbo",
    },
    {
        "suffix": "positive_clip_normalized",
        "cam_key": "clip_normalized_cams",
        "overlay_name": "positive clip-normalized (cross-frame magnitude comparison only)",
        "signed": False,
        "positive_display_mode": "enhanced",
        "cmap_name": "turbo",
    },
    {
        "suffix": "signed_clip_normalized_faithful",
        "cam_key": "signed_clip_normalized_cams",
        "overlay_name": "signed clip-normalized faithful",
        "signed": True,
        "signed_display_mode": "faithful",
        "signed_display_percentile": None,
        "cmap_name": "coolwarm",
    },
    {
        "suffix": "signed_clip_normalized_enhanced_visualization_only",
        "cam_key": "signed_clip_normalized_cams",
        "overlay_name": "signed clip-normalized enhanced visualization only",
        "signed": True,
        "signed_display_mode": "enhanced",
        "signed_display_percentile": None,
        "cmap_name": "coolwarm",
    },
    {
        "suffix": "signed_robust_display_enhanced_visualization_only",
        "cam_key": "signed_clip_normalized_cams",
        "overlay_name": "signed robust-display enhanced visualization only",
        "signed": True,
        "signed_display_mode": "enhanced",
        "signed_display_percentile": 99.0,
        "cmap_name": "coolwarm",
    },
]

for model_name, model in models.items():
    model_root = RUN_DIR / model_name
    for sample_id in tqdm(selected_by_model[model_name], desc=f"Grad-CAM {model_name}"):
        idx = sample_id_to_index[sample_id]
        batch = next(iter(DataLoader(Subset(test_dataset, [idx]), batch_size=1, shuffle=False, num_workers=0)))
        sequence = batch["sequence"].to(device)
        assert sequence.shape == (1, int(BASE_CONFIG["sequence_length"]), 1, *tuple(BASE_CONFIG["image_size"])), tuple(sequence.shape)

        pred_row = prediction_tables[model_name][prediction_tables[model_name]["sample_id"] == sample_id].iloc[0]

        # For temporal analysis against optical flow, encoder_bottleneck is the faithful primary CAM type.
        # temporal_representation maps are secondary counterfactual EF-head probes, not ground truth per-frame contribution.
        cam_jobs = [
            (
                "encoder_bottleneck",
                "bottleneck_encoder",
                encoder_bottleneck_ef_gradcam(model, sequence, ef_mean=ef_mean, ef_std=ef_std),
            ),
            (
                "temporal_representation",
                "fused_bidirectional_temporal_representation",
                temporal_representation_ef_probe_gradcam(model, sequence, ef_mean=ef_mean, ef_std=ef_std),
            ),
        ]

        for cam_type, target_layer, result in cam_jobs:
            assert abs(result.pred_ef - float(pred_row["ef_pred"])) < 1e-3, (result.pred_ef, float(pred_row["ef_pred"]))
            cam_dir = model_root / cam_type
            npz_path = cam_dir / "npz" / f"{sample_id}_{cam_type}_gradcam.npz"
            diagnostics_csv = cam_dir / "diagnostics" / f"{sample_id}_{cam_type}_diagnostics.csv"
            row = save_gradcam_npz(
                npz_path,
                result,
                batch,
                model_name=model_name,
                checkpoint_metadata=checkpoint_metadata[model_name],
                cam_type=cam_type,
                target_layer=target_layer,
                dataset_metrics=dataset_metrics[model_name],
                diagnostics_csv_path=diagnostics_csv,
            )

            overlay_paths = []
            for spec in overlay_specs:
                overlay_path = cam_dir / "overlays" / spec["suffix"] / f"{sample_id}_{cam_type}_{spec['suffix']}_overlay.png"
                make_gradcam_overlay_figure(
                    npz_path,
                    overlay_path,
                    dataset_metrics[model_name],
                    cam_key=spec["cam_key"],
                    overlay_name=spec["overlay_name"],
                    signed=bool(spec["signed"]),
                    positive_display_mode=spec.get("positive_display_mode", "faithful"),
                    signed_display_mode=spec.get("signed_display_mode", "faithful"),
                    signed_display_percentile=spec.get("signed_display_percentile"),
                    cmap_name=spec.get("cmap_name"),
                    alpha=float(BASE_CONFIG["overlay_alpha"]),
                )
                overlay_paths.append(str(overlay_path))

            diagnostic_plot_path = cam_dir / "diagnostic_plots" / f"{sample_id}_{cam_type}_temporal_diagnostics.png"
            make_temporal_diagnostic_plot(
                diagnostics_csv,
                diagnostic_plot_path,
                target_idx=int(batch["target_idx"][0]),
                title=f"{model_name} | {cam_type} | {sample_id}",
            )

            motion_trace_path = cam_dir / "motion_trace_overlays" / "frame_normalized_primary" / f"{sample_id}_{cam_type}_frame_normalized_motion_trace_overlay.png"
            centroid_df = make_motion_trace_overlay_figure(
                npz_path,
                motion_trace_path,
                dataset_metrics[model_name],
                cam_key="frame_normalized_positive_cams",
                overlay_name="frame-normalized motion trace (primary)",
                alpha=float(BASE_CONFIG["overlay_alpha"]),
                positive_display_mode="enhanced",
            )
            centroid_df["centroid_source"] = "frame_normalized_positive_cams"

            magnitude_motion_trace_path = cam_dir / "motion_trace_overlays" / "clip_normalized_magnitude_comparison" / f"{sample_id}_{cam_type}_clip_normalized_motion_trace_overlay.png"
            magnitude_centroid_df = make_motion_trace_overlay_figure(
                npz_path,
                magnitude_motion_trace_path,
                dataset_metrics[model_name],
                cam_key="clip_normalized_cams",
                overlay_name="clip-normalized motion trace (cross-frame magnitude comparison only)",
                alpha=float(BASE_CONFIG["overlay_alpha"]),
                positive_display_mode="enhanced",
            )
            magnitude_centroid_df["centroid_source"] = "clip_normalized_cams_magnitude_comparison"
            overlay_paths.append(str(motion_trace_path))
            overlay_paths.append(str(magnitude_motion_trace_path))
            centroid_csv = cam_dir / "centroid_diagnostics" / f"{sample_id}_{cam_type}_centroids.csv"
            centroid_csv.parent.mkdir(parents=True, exist_ok=True)
            centroid_df.insert(0, "sample_id", sample_id)
            centroid_df.insert(1, "video_id", str(batch["video_id"][0]))
            centroid_df.insert(2, "model_name", model_name)
            centroid_df.insert(3, "cam_type", cam_type)
            centroid_df["target_idx"] = int(batch["target_idx"][0])
            centroid_df["target_frame_idx"] = int(batch["frame_idx"][0])
            centroid_df["npz_path"] = str(npz_path)
            centroid_df.to_csv(centroid_csv, index=False)

            centroid_plot_path = cam_dir / "centroid_plots" / f"{sample_id}_{cam_type}_centroid_trajectory.png"
            make_centroid_trajectory_plot(
                centroid_csv,
                centroid_plot_path,
                target_idx=int(batch["target_idx"][0]),
                title=f"{model_name} | {cam_type} | {sample_id}",
            )

            row["overlay_paths"] = json.dumps(overlay_paths)
            row["primary_overlay_path"] = overlay_paths[0]
            row["diagnostic_plot_path"] = str(diagnostic_plot_path)
            row["motion_trace_overlay_path"] = str(motion_trace_path)
            row["motion_trace_overlay_path_magnitude_comparison"] = str(magnitude_motion_trace_path)
            row["centroid_csv_path"] = str(centroid_csv)
            row["centroid_plot_path"] = str(centroid_plot_path)
            manifest_rows.append(row)

            diagnostics_df = pd.read_csv(diagnostics_csv)
            diagnostics_df["npz_path"] = str(npz_path)
            diagnostics_df["diagnostic_plot_path"] = str(diagnostic_plot_path)
            all_diagnostic_rows.append(diagnostics_df)
            all_centroid_rows.append(centroid_df)

cam_manifest_df = pd.DataFrame(manifest_rows)
cam_manifest_df.to_csv(MANIFEST_DIR / "ef_gradcam_manifest.csv", index=False)
all_temporal_diagnostics_df = pd.concat(all_diagnostic_rows, ignore_index=True) if all_diagnostic_rows else pd.DataFrame()
all_temporal_diagnostics_df.to_csv(MANIFEST_DIR / "ef_gradcam_temporal_diagnostics.csv", index=False)
all_centroid_diagnostics_df = pd.concat(all_centroid_rows, ignore_index=True) if all_centroid_rows else pd.DataFrame()
all_centroid_diagnostics_df.to_csv(MANIFEST_DIR / "ef_gradcam_centroid_diagnostics.csv", index=False)
display(cam_manifest_df.head())


## Required Outputs

In [ ]:
required_outputs = [
    RUN_DIR / "config.json",
    MANIFEST_DIR / "checkpoint_metadata.json",
    MANIFEST_DIR / "normal_test_metrics.json",
    MANIFEST_DIR / "selected_gradcam_samples.csv",
    MANIFEST_DIR / "ef_gradcam_manifest.csv",
    MANIFEST_DIR / "ef_gradcam_temporal_diagnostics.csv",
    MANIFEST_DIR / "ef_gradcam_centroid_diagnostics.csv",
]
for model_name in models:
    required_outputs.extend([
        MANIFEST_DIR / f"{model_name}_normal_test_predictions.csv",
        RUN_DIR / model_name / "encoder_bottleneck" / "npz",
        RUN_DIR / model_name / "encoder_bottleneck" / "overlays",
        RUN_DIR / model_name / "temporal_representation" / "npz",
        RUN_DIR / model_name / "temporal_representation" / "overlays",
    ])
missing = [str(path) for path in required_outputs if not path.exists()]
assert not missing, f"Missing expected outputs: {missing}"
print(f"All EF Grad-CAM outputs saved under: {RUN_DIR}")
print(cam_manifest_df.groupby(["model_name", "cam_type"]).size())
